# 🧠 Web Log Analyzer — ML Model Visualization Notebook

This notebook loads the saved ML models, results, and graphs:

### ✔ Accuracy Comparison Graph
### ✔ Confusion Matrices (RF + XGBoost)
### ✔ Classification Reports
### ✔ Feature Importances (RF, XGB)
### ✔ Reload and Test Trained Models

---

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import joblib
from PIL import Image

import warnings
warnings.filterwarnings('ignore')

## 📥 Load Saved Models

In [2]:
rf = joblib.load('../models/ml/random_forest.pkl')
xgb = joblib.load('../models/ml/xgboost.pkl')
iso = joblib.load('../models/ml/iso_forest.pkl')

print("Models loaded successfully!")

Models loaded successfully!


## 📥 Load Feature Dataset (Step 2 Output)

In [9]:
print(df.columns)


Index(['ip', 'timestamp', 'method', 'url', 'protocol', 'status', 'size',
       'referer', 'agent', 'raw_request', 'hour', 'minute', 'day', 'weekday',
       'is_weekend', 'path', 'path_length', 'path_depth', 'query',
       'query_params', 'file_type', 'is_bot', 'ip_numeric', 'prev_time',
       'time_diff', 'new_session', 'session_id'],
      dtype='object')


In [3]:
df = pd.read_parquet('../data/features/apache_features.parquet')
df.head()

,ip,timestamp,method,url,protocol,status,size,referer,agent,raw_request,...,path_depth,query,query_params,file_type,is_bot,ip_numeric,prev_time,time_diff,new_session,session_id
0,1.132.107.223,2019-01-26 02:26:22+03:30,GET,/m/article/616/%D8%B9%D9%84%D8%AA-%D8%AE%D9%88...,HTTP/1.1,200,19688,https://www.google.com/,Mozilla/5.0 (Linux; Android 8.0.0; SAMSUNG SM-...,GET /m/article/616/%D8%B9%D9%84%D8%AA-%D8%AE%D...,...,4,,0,,False,25455583,NaT,999999.0,True,1
1,1.132.107.223,2019-01-26 02:26:23+03:30,GET,/settings/logo,HTTP/1.1,200,4120,https://www.zanbil.ir/m/article/616/%D8%B9%D9%...,Mozilla/5.0 (Linux; Android 8.0.0; SAMSUNG SM-...,GET /settings/logo HTTP/1.1,...,2,,0,,False,25455583,2019-01-26 02:26:22+03:30,1.0,False,1
2,1.132.107.223,2019-01-26 02:26:24+03:30,GET,/amp-helper-frame.html?appId=a624a1c1-0c93-466...,HTTP/1.1,200,133,https://www.zanbil.ir/m/article/616/%D8%B9%D9%...,Mozilla/5.0 (Linux; Android 8.0.0; SAMSUNG SM-...,GET /amp-helper-frame.html?appId=a624a1c1-0c93...,...,1,appId=a624a1c1-0c93-466a-a546-e146710f97e6&par...,2,html,False,25455583,2019-01-26 02:26:23+03:30,1.0,False,1
3,1.132.107.223,2019-01-26 02:26:25+03:30,GET,/static/images/guarantees/bestPrice.png,HTTP/1.1,200,7356,https://www.zanbil.ir/m/article/616/%D8%B9%D9%...,Mozilla/5.0 (Linux; Android 8.0.0; SAMSUNG SM-...,GET /static/images/guarantees/bestPrice.png HT...,...,4,,0,png,False,25455583,2019-01-26 02:26:24+03:30,1.0,False,1
4,1.132.107.223,2019-01-26 02:26:25+03:30,GET,/static/images/guarantees/goodShopping.png,HTTP/1.1,200,6496,https://www.zanbil.ir/m/article/616/%D8%B9%D9%...,Mozilla/5.0 (Linux; Android 8.0.0; SAMSUNG SM-...,GET /static/images/guarantees/goodShopping.png...,...,4,,0,png,False,25455583,2019-01-26 02:26:25+03:30,0.0,False,1


In [10]:
import numpy as np

df['label'] = (
    (df['status'] >= 400) |        # Errors
    (df['path_length'] > 60) |    # Very long URLs
    (df['time_diff'] < 0.2) |     # Too fast requests (bot behavior)
    (df['is_bot'] == 1)           # Known bots
).astype(int)


## ⚙ Build X and y

In [11]:
# Features (same as used in ML)
features = df.drop(['label'], axis=1)
target = df['label']

X = features
y = target

print(X.shape, y.shape)

(10365108, 27) (10365108,)


## 🔍 Evaluate RandomForest Again (In Notebook)

In [12]:
preds_rf = rf.predict(X)
print(classification_report(y, preds_rf))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- agent
- day
- file_type
- ip
- is_bot
- ...


### Confusion Matrix — RandomForest

In [13]:
cm = confusion_matrix(y, preds_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — RandomForest')
plt.show()

NameError: name 'preds_rf' is not defined

## 🔍 Evaluate XGBoost Again (In Notebook)

In [ ]:
preds_xgb = xgb.predict(X)
print(classification_report(y, preds_xgb))

### Confusion Matrix — XGBoost

In [ ]:
cm = confusion_matrix(y, preds_xgb)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.title('Confusion Matrix — XGBoost')
plt.show()

## 📊 Feature Importance — RandomForest

In [ ]:
importances = rf.feature_importances_
feat_names = X.columns

imp_df = pd.DataFrame({'Feature': feat_names, 'Importance': importances})
imp_df = imp_df.sort_values('Importance', ascending=False).head(20)

sns.barplot(x='Importance', y='Feature', data=imp_df)
plt.title('Top 20 Important Features — RF')
plt.show()

## 📊 Feature Importance — XGBoost

In [ ]:
xgb_importances = xgb.feature_importances_

imp_df = pd.DataFrame({'Feature': feat_names, 'Importance': xgb_importances})
imp_df = imp_df.sort_values('Importance', ascending=False).head(20)

sns.barplot(x='Importance', y='Feature', data=imp_df, palette='magma')
plt.title('Top 20 Important Features — XGBoost')
plt.show()

## 🖼 Show Saved PNG Graphs (Generated During Training)

In [ ]:
def show_image(path):
    img = Image.open(path)
    plt.imshow(img)
    plt.axis('off')
    plt.show()

print("Showing saved graphs...")

show_image('../results/ml/accuracy_comparison.png')
show_image('../results/ml/RandomForest_confusion_matrix.png')
show_image('../results/ml/XGBoost_confusion_matrix.png')